# Adaptive RAG — Quick Implementation

## Problem Statement
> HR gets flooded with the same policy questions every day (leave, WFH stipends, travel limits...).
> Build an **Adaptive RAG** assistant that reads the 5 company policy files in `company_docs/`,
> and for **each question** decides the right strategy:
>
> - **Policy question** → retrieve relevant docs, grade them, answer — and if retrieval fails,
>   **rewrite the question** and retry before giving up.
> - **Off-topic question** → don't hallucinate; reply that it's outside company policy.

That's what makes it *adaptive*: the pipeline adapts to the question, instead of running
one fixed retrieve-and-generate path for everything.

## Pipeline
```
START ──► ROUTER ──(off-topic)──► "not covered by policy" reply ──► END
              │(policy question)
              ▼
        RETRIEVE + GRADE docs
              │ relevant                    │ nothing relevant
              ▼                             ▼
          GENERATE ──► END        REWRITE (max 2 tries) ──► RETRIEVE again
                                          │ still nothing
                                          ▼
                              "not covered by policy" reply ──► END
```

## Stack (2026, no legacy/deprecated modules)
`langchain-core` · `langchain-openai` · `langchain-community` (FAISS) · `langchain-text-splitters` · `langgraph`

## 1. Setup & Imports

Everything below uses the current LangChain ecosystem. Nothing from the old
`langchain.chains` / `langchain_experimental` era.

In [4]:
import os
os.environ.setdefault("OPENAI_API_KEY", "sk-your-key-here")

from typing import Literal, TypedDict

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langgraph.graph import StateGraph, START, END

from pathlib import Path

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 2. Load the Knowledge Base

Load the five policy `.txt` files, chunk them, and index into a local FAISS vector store.

In [ ]:
KB_DIR = Path("/Users/pradipwasre/GenAI-V2/Langgraph/adaptive_rag")

docs = [Document(page_content=p.read_text(encoding="utf-8"), metadata={"source": p.name})
        for p in sorted(KB_DIR.glob("*.txt"))]
print(f"Loaded {len(docs)} docs: {[d.metadata['source'] for d in docs]}")

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(docs)
print(f"Chunks: {len(chunks)}")

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

## 3. The LLM Decision Helpers

Adaptive RAG is just a few small, single-purpose LLM calls:

1. **Router** — is this question about company policy, or not?
2. **Retrieval Grader** — is a retrieved document actually relevant?
3. **Question Rewriter** — rephrase a failed question into a better retrieval query.
4. **Generator** — produce the final answer from the graded documents.

In [ ]:
# 1. ROUTER — classify the question
router_chain = ChatPromptTemplate.from_messages([
    ("system", """Classify the question into exactly one of:
- "policy": about company policies (leave, work from home, travel, handbook, insurance)
- "other": anything else (news, sports, general chat, coding, etc.)
Reply with ONLY one word: policy or other."""),
    ("human", "{question}"),
]) | llm | StrOutputParser()

def route_question(state) -> Literal["policy", "other"]:
    out = router_chain.invoke({"question": state["question"]}).strip().lower()
    return "policy" if "policy" in out else "other"

# 2. RETRIEVAL GRADER — is a doc relevant to the question?
grader_chain = ChatPromptTemplate.from_messages([
    ("system", """You are a grader. Reply "yes" if the document is relevant to the question,
otherwise reply "no". One word only."""),
    ("human", "Document:\n{document}\n\nQuestion: {question}"),
]) | llm | StrOutputParser()

# 3. QUESTION REWRITER — improve a question that failed retrieval
rewriter_chain = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the question into a clearer, more searchable version for policy retrieval. Reply with ONLY the rewritten question."),
    ("human", "{question}"),
]) | llm | StrOutputParser()

# 4. GENERATOR — answer from the retrieved context
rag_chain = ChatPromptTemplate.from_messages([
    ("system", """Answer the question using only the context below. Keep it under 4 sentences.
If the context does not contain the answer, say "This is not covered in the company policy documents."

Context:
{context}"""),
    ("human", "{question}"),
]) | llm | StrOutputParser()

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

## 4. Graph State & Nodes

State carries everything the graph needs between steps. Each node reads the state
and returns only what changed.

- **retrieve** — fetches docs, keeps only the ones the grader marks relevant.
- **rewrite_or_giveup** — rewrites the question for another retrieval try (max 2);
  if retries are exhausted, it produces the "not covered" fallback answer itself.
- **off_topic_reply** — the adaptive refusal path for non-policy questions.

In [ ]:
class State(TypedDict):
    question: str
    documents: list[Document]
    generation: str
    retries: int

def retrieve(state: State):
    """Retrieve docs from FAISS and keep only grader-approved ones."""
    found = retriever.invoke(state["question"])
    relevant = [d for d in found
                if grader_chain.invoke({"document": d.page_content,
                                        "question": state["question"]}).strip().lower() == "yes"]
    return {"documents": relevant}

def rewrite_or_giveup(state: State):
    """Nothing relevant found: rewrite & retry, or give a fallback answer."""
    if state["retries"] < 2:
        better = rewriter_chain.invoke({"question": state["question"]}).strip()
        return {"question": better, "retries": state["retries"] + 1}
    return {"generation": "This question is not covered in the company policy documents.",
            "documents": []}

def off_topic_reply(state: State):
    """Adaptive refusal — the router sent us here, no retrieval at all."""
    return {"generation": "I'm a company-policy assistant, so I can only answer questions "
                          "about leave, remote work, travel, and insurance policies."}

def generate(state: State):
    """Answer from the graded documents."""
    return {"generation": rag_chain.invoke({"context": format_docs(state["documents"]),
                                            "question": state["question"]})}

## 5. Assemble the Graph

Wiring:
- `START` → router decides: policy question goes to **retrieve**, anything else to **off_topic_reply**.
- After **retrieve**: relevant docs found → **generate**; nothing → **rewrite_or_giveup**.
- After **rewrite_or_giveup**: if it produced a question, loop back to **retrieve**;
  if it produced a fallback answer, go to **END**.

In [ ]:
def after_retrieve(state: State) -> Literal["generate", "rewrite"]:
    return "generate" if state["documents"] else "rewrite"

def after_rewrite(state: State) -> Literal["retrieve", "end"]:
    return "end" if state["generation"] else "retrieve"   # fallback answer set => done

builder = StateGraph(State)
builder.add_node("retrieve", retrieve)
builder.add_node("rewrite_or_giveup", rewrite_or_giveup)
builder.add_node("off_topic_reply", off_topic_reply)
builder.add_node("generate", generate)

builder.add_conditional_edges(START, route_question, {
    "policy": "retrieve",
    "other": "off_topic_reply",
})
builder.add_conditional_edges("retrieve", after_retrieve, {
    "generate": "generate",
    "rewrite": "rewrite_or_giveup",
})
builder.add_conditional_edges("rewrite_or_giveup", after_rewrite, {
    "retrieve": "retrieve",
    "end": END,
})
builder.add_edge("off_topic_reply", END)
builder.add_edge("generate", END)

graph = builder.compile()
graph

## 6. Run It — See the Adaptivity

Three questions, three different paths through the same graph:

**Q1 — policy question, easy retrieval** → router → retrieve (docs pass) → generate

In [ ]:
r1 = graph.invoke({"question": "How much is the work-from-home internet stipend?",
                   "documents": [], "generation": "", "retries": 0})
print(r1["generation"])

**Q2 — policy question, vague wording** → retrieve fails → rewrite → retrieve succeeds → generate

In [ ]:
r2 = graph.invoke({"question": "money for setting up home office",
                   "documents": [], "generation": "", "retries": 0})
print(r2["generation"])

**Q3 — off-topic question** → router sends it straight to the refusal path (no retrieval, no hallucination)

In [ ]:
r3 = graph.invoke({"question": "Who won the World Cup 2026?",
                   "documents": [], "generation": "", "retries": 0})
print(r3["generation"])

## 7. Recap

| Question | Path taken | Why it's adaptive |
|---|---|---|
| WFH stipend | retrieve → generate | relevant docs found first try |
| "money for home office" | retrieve ✗ → rewrite → retrieve ✓ → generate | question improved before retrying |
| World Cup | off_topic_reply | no retrieval, no hallucination |

This is the core of Adaptive RAG: **route first, grade what you retrieve, rewrite before
you give up, and refuse gracefully when the knowledge base can't help.**

**Production upgrades when you need them:** `with_structured_output` for the graders,
a hallucination + answer grader after generation, LangSmith tracing, and a
`MemorySaver` checkpointer for multi-turn chat.